# Phase 2: Intelligent Victorian Chatbot
**Architecture**: Rules + Dialogue Memory + Best-of-K Multi-Criteria Re-Ranking
**Corpus**: Jane Austen's *Pride and Prejudice*

---

## 1. Pillars of Intelligence
1. **Dialogue Act / Intent Classification**: Instant in-character persona handling for greetings, identity queries, farewells, and opinions.
2. **Conversational Memory & Coreference Resolution**: Remembers the active entity (`last_entity`) so follow-up pronouns like *"is he proud?"* automatically resolve to Darcy.
3. **Best-of-K Candidate Generation & Re-Ranking**: Generates $K=5$ candidate continuations and scores them on relevance, fluency, dangling preposition penalties, and repetition.
4. **ELIZA-Style Pronoun Reflection**: Reflects user first-person statements (*"I feel"* $\to$ *"You feel"*).


In [1]:
import sys
from pathlib import Path
root = Path.cwd()
while root.name and not (root / "src").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.chatbot import IntelligentChatbot

bot = IntelligentChatbot(corpus_name="pride", n=3, temperature=0.8, seed=42)
print("IntelligentChatbot initialized successfully.")


IntelligentChatbot initialized successfully.


## 2. Multi-Turn Coreference & Intent Demonstration


In [1]:
dialogue = [
    "Good morning! How do you do?",
    "What do you think of Mr. Darcy?",
    "Is he proud?",                         # 'he' -> resolved to Darcy
    "I feel rather fatigued today.",
    "Farewell, good day!",
]

for i, p in enumerate(dialogue, 1):
    reply = bot.respond(p)
    print(f"Turn {i}:")
    print(f"  User  : {p}")
    print(f"  Austen: {reply}")
    print(f"  [Memory]: Last Entity = {bot.last_entity} | Intent = {bot.last_intent}\n")


Turn 1:
  User  : Good morning! How do you do?
  Austen: Good day to you! I trust you find yourself in tolerable health.
  [Memory]: Last Entity = None | Intent = GREETING

Turn 2:
  User  : What do you think of Mr. Darcy?
  Austen: As for Mr. Darcy, I must confess that Darcy they had been in the morning s walk they had.
  [Memory]: Last Entity = darcy | Intent = OPINION

Turn 3:
  User  : Is he proud?
  Austen: Darcy s character was decided.
  [Memory]: Last Entity = darcy | Intent = None

Turn 4:
  User  : I feel rather fatigued today.
  Austen: You say you feel rather fatigued today? Well, feel it all.
  [Memory]: Last Entity = darcy | Intent = None

Turn 5:
  User  : Farewell, good day!
  Austen: I take my leave of you with the utmost civility. Farewell!
  [Memory]: Last Entity = darcy | Intent = FAREWELL


## Observations
- **Pronoun Resolution**: In Turn 3, *"Is he proud?"* seamlessly resolved `"he"` to `"darcy"`.
- **Social Turns**: Greetings and Farewells stay perfectly in-character instead of generating disjointed n-gram text.
